# FloodShift 04H — Single-Model Hierarchical Water Head

**Purpose:** one final, tightly controlled *training* experiment before the frozen geographic test is opened.

04G showed that the problem is naturally hierarchical:

1. **water vs non-water**
2. **permanent vs temporary, conditioned on water**

04H turns that observation into **one deployable neural model**. It keeps the proven 04E TerraMind-RIFT segmenter frozen and learns only a small, zero-initialized hierarchical residual head.

## Scientific stop rules

- **No test evaluation exists in this notebook.**
- The Pakistan + Spain frozen test remains locked.
- No TerraMind, RIFT, router, decoder, preprocessing, split, or target semantics are changed.
- No hyperparameter sweep is performed.
- Exactly **6 head-only epochs** are run with a predeclared loss and optimizer.
- Validation uses the same **4-way TTA** protocol as 04E/04F/04G.
- If 04H does not meet the predeclared gate, do **not** invent 04I automatically; inspect the result first.

## Required Kaggle inputs

**Required**
- Frozen Notebook-03 preprocessing output containing:
  - `floodshift_preprocessing_manifest_v1.csv`
  - `floodshift_preprocessing_GLOBAL_LOCK_v1.json`
  - `optical_10band_train_normalization_exact.csv`
  - `train_raw_sar_scene_statistics.csv`
  - the raster / reliability files referenced by the manifest
- Exact 04E checkpoint:
  - `FloodShift_FINAL_TerraMind_RIFT_best.pt`
  - expected SHA256: `6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822`

**Optional lineage/audit input**
- `FloodShift_04F_best.pt`
  - expected SHA256: `75f787419da48c8ce660a0ffafe5a7835c420ec9cd8a06f1d9623e3012397aac`
  - **04F weights are NOT used to train 04H.**
- `FloodShift_04G_decision_record.json` is optional; 04G reference metrics are frozen below.

You do **not** need the 04E or 04F `.ipynb` notebooks themselves. Their required model/data reconstruction code is contained here.


In [ ]:
# ============================================================
# CELL: P10-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P10-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [ ]:
# ============================================================
# CELL: P10-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
# Deliberately use one T4, matching the proven 04E/04F/04G path.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, re, json, time, math, random, copy, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_04H_Hierarchical_Single_Model")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
BOOTSTRAP_ROOT = OUTPUT_ROOT / "bootstrap"

for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, METRIC_ROOT, BOOTSTRAP_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P10-C01 — 04H ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P10-C02 — LOCATE FROZEN PREPROCESSING + 04E
# 04F is optional and audit-only.
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)

    if cache_key in _file_cache:
        return _file_cache[cache_key]

    roots = [INPUT_ROOT, BOOTSTRAP_ROOT]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input or bootstrap."
            )
        return None

    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[cache_key] = selected
    return selected

# Auto-extract small experiment archives if attached instead of direct .pt files.
for archive_name in [
    "FloodShift_FINAL_ARCHIVE.zip",
    "FloodShift_04F_ARCHIVE.zip",
]:
    matches = list(INPUT_ROOT.rglob(archive_name))
    if matches:
        archive = sorted(matches, key=lambda p: len(str(p)))[0]
        print("Extracting:", archive)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(BOOTSTRAP_ROOT)

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04E = locate_by_saved_path("FloodShift_FINAL_TerraMind_RIFT_best.pt")
CHECKPOINT_04F = locate_by_saved_path("FloodShift_04F_best.pt", required=False)
DECISION_04G = locate_by_saved_path("FloodShift_04G_decision_record.json", required=False)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P10-C02 — INPUTS FOUND")
print("=" * 112)
for p in [
    FINAL_MANIFEST_PATH,
    GLOBAL_LOCK_PATH,
    OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH,
    CHECKPOINT_04E,
]:
    print(" ", p)

print("04F audit-only :", CHECKPOINT_04F if CHECKPOINT_04F else "not attached (OK)")
print("04G record     :", DECISION_04G if DECISION_04G else "not attached (OK)")
print("Lock           :", preprocessing_lock["status"])
print("P10-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P10-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

In [ ]:
# ============================================================
# CELL: P10-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [ ]:
# ============================================================
# CELL: P10-C05 — TRAIN + VALIDATION LOADERS
#
# Frozen test split is asserted but NO test Dataset/DataLoader is created.
# ============================================================
TRAIN_BATCH_SIZE = 2
VAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_dataset = FixedEvalDataset(train_df, train_mode=True)
val_dataset = FixedEvalDataset(val_df, train_mode=False)

common = dict(
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    generator=g,
    **common,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    **common,
)

batch = next(iter(val_loader))

print("=" * 112)
print("P10-C05 — DATA LOADER CHECK")
print("=" * 112)
print("Train scenes / tiles:", len(train_df), "/", len(train_dataset))
print("Val scenes / tiles  :", len(val_df), "/", len(val_dataset))
print("Frozen test scenes  :", len(test_df), "(LOCKED; no loader constructed)")
print("Train batch         :", TRAIN_BATCH_SIZE)
print("Grad accumulation   :", GRAD_ACCUM_STEPS)
print("Effective batch     :", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("SAR                 :", tuple(batch["sar"].shape))
print("Optical             :", tuple(batch["optical"].shape))
print("Reliability         :", tuple(batch["reliability"].shape))
print("Target              :", tuple(batch["target"].shape))

assert batch["sar"].shape[1:] == (2, 256, 256)
assert batch["optical"].shape[1:] == (10, 256, 256)
assert len(test_df) == 58
print("P10-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [ ]:
# ============================================================
# CELL: P10-C07 — LOAD EXACT 04E + BUILD ZERO-INIT 04H HEAD
# ============================================================
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_04F_SHA256 = "75f787419da48c8ce660a0ffafe5a7835c420ec9cd8a06f1d9623e3012397aac"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

sha04e = sha256_file(CHECKPOINT_04E)
print("04E SHA256:", sha04e)
assert sha04e == EXPECTED_04E_SHA256, "Wrong 04E checkpoint."

sha04f = None
if CHECKPOINT_04F is not None:
    sha04f = sha256_file(CHECKPOINT_04F)
    print("04F SHA256 (audit only):", sha04f)
    if sha04f != EXPECTED_04F_SHA256:
        print("⚠️ Attached 04F hash differs from the inspected 04F. This does NOT affect 04H training.")

def sanitize_known_unused(state):
    state = dict(state)
    if "backbone.missing_mod_token" in state:
        t = state["backbone.missing_mod_token"]
        if torch.is_floating_point(t) and not torch.isfinite(t).all():
            state["backbone.missing_mod_token"] = torch.zeros_like(t)
    return state

def load_exact_04e(path):
    base = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    state = sanitize_known_unused(state)

    missing, unexpected = base.load_state_dict(state, strict=False)
    assert not missing, f"04E missing keys: {missing[:10]}"
    assert not unexpected, f"04E unexpected keys: {unexpected[:10]}"

    bad = [
        n for n, t in base.state_dict().items()
        if torch.is_floating_point(t) and not torch.isfinite(t).all()
    ]
    assert not bad, f"04E non-finite tensors: {bad[:10]}"
    return base

class HierarchicalResidualHead(nn.Module):
    """
    Input = [04E base probabilities (3), SAR (2), optical (10), P(clear) (1)] = 16 channels.
    Output = two bounded residuals:
      delta_water : correction to water-vs-background logit
      delta_type  : correction to temporary-vs-permanent logit
    """
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        # Exact identity at initialization.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        # Bounded but locally identity-gradient residual parameterization.
        delta = self.max_delta * torch.tanh(raw / self.max_delta)
        return delta

class FloodShift04H(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()

        # 04E stays permanently frozen in 04H.
        for p in self.base.parameters():
            p.requires_grad = False
        self.base.eval()

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)
        # Never allow the frozen 04E backbone/decoder to enter train mode.
        self.base.eval()
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        # No gradient graph through 04E.
        with torch.no_grad():
            base_out = self.base(sar, optical, reliability, return_aux=True)
            base_logits = base_out["logits"].float()
            base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )
        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        # Exact hierarchical decomposition of a 3-way softmax:
        # water-vs-background = logsumexp(z_perm,z_temp) - z_nonflood
        # temporary-vs-permanent = z_temp - z_perm
        base_water_logit = torch.logsumexp(base_logits[:, 1:3], dim=1) - base_logits[:, 0]
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,      # log-probabilities; softmax(logits) == probs
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "base_water_logit": base_water_logit,
                "base_type_logit": base_type_logit,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
            }
        return log_probs

base_04e = load_exact_04e(CHECKPOINT_04E).to(DEVICE).eval()
model = FloodShift04H(base_04e).to(DEVICE)

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable_n = sum(n for _, n in trainable)

print("=" * 112)
print("P10-C07 — MODEL BUILD")
print("=" * 112)
print("Frozen parameters   :", f"{frozen:,}")
print("Trainable parameters:", f"{trainable_n:,}")
print("Trainable tensors   :", [n for n, _ in trainable])
assert all(name.startswith("head.") for name, _ in trainable)

# Mandatory exact identity audit at initialization.
model.eval()
with torch.inference_mode():
    sar = batch["sar"][:1].to(DEVICE)
    opt = batch["optical"][:1].to(DEVICE)
    rel = batch["reliability"][:1].to(DEVICE)

    base_logits = base_04e(sar, opt, rel)
    h_out = model(sar, opt, rel, return_aux=True)

    base_prob = torch.softmax(base_logits.float(), dim=1)
    max_prob_diff = (base_prob - h_out["probs"]).abs().max().item()
    pred_diff = (base_prob.argmax(1) != h_out["probs"].argmax(1)).sum().item()
    max_delta = max(
        h_out["delta_water"].abs().max().item(),
        h_out["delta_type"].abs().max().item(),
    )

print("Identity max probability diff:", f"{max_prob_diff:.3e}")
print("Identity prediction diff px  :", pred_diff)
print("Initial max residual         :", f"{max_delta:.3e}")

assert max_prob_diff < 2e-6, "04H is not an identity warm-start of 04E."
assert pred_diff == 0
assert max_delta == 0.0
print("P10-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C08 — TRAIN-SET TARGET CENSUS + PREDECLARED LOSS
# ============================================================
# Count labels from TRAIN ONLY. Validation and test statistics are not used.
train_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
train_ignore = 0

start = time.time()
for i, row in train_df.iterrows():
    paths = resolve_scene_files(row)

    with rasterio.open(paths["flood"]) as src:
        flood = src.read(1)

    with rasterio.open(paths["jrc"]) as src:
        jrc = src.read(1)

    target = build_three_class_target(flood, jrc)

    valid = target != IGNORE_INDEX
    train_ignore += int((~valid).sum())
    train_counts += np.bincount(
        target[valid].astype(np.int64),
        minlength=NUM_CLASSES,
    )

    if (i + 1) % 50 == 0 or (i + 1) == len(train_df):
        print(f"[{i+1:03d}/{len(train_df)}] label census")

n0, n1, n2 = [float(x) for x in train_counts]
n_water = n1 + n2

# Conservative square-root balancing for water/background.
WATER_POS_WEIGHT = float(np.clip(np.sqrt(n0 / max(n_water, 1.0)), 1.0, 4.0))

# In the type BCE, positive = temporary and negative = permanent.
# This balances aggregate permanent/temporary contribution without over-pushing temporary water.
TYPE_TEMP_POS_WEIGHT = float(np.clip(n1 / max(n2, 1.0), 0.10, 10.0))

print("=" * 112)
print("P10-C08 — TRAIN-ONLY CLASS COUNTS")
print("=" * 112)
print("non-flood       :", int(n0))
print("permanent water :", int(n1))
print("temporary flood :", int(n2))
print("ignore          :", int(train_ignore))
print("water pos weight:", round(WATER_POS_WEIGHT, 4))
print("type temp weight:", round(TYPE_TEMP_POS_WEIGHT, 4))
print("Census minutes  :", round((time.time() - start) / 60, 2))

# Predeclared loss weights. No sweep.
LOSS_W_3CLASS = 0.50
LOSS_W_WATER = 0.75
LOSS_W_TYPE = 1.00
LOSS_W_ANCHOR = 0.15
ANCHOR_CONFIDENCE = 0.90

def hierarchical_loss(outputs, target):
    log_probs = outputs["logits"].float()
    base_prob = outputs["base_prob"].float()
    water_logit = outputs["water_logit"].float()
    type_logit = outputs["type_logit"].float()

    valid = target != IGNORE_INDEX
    water_mask = valid & ((target == 1) | (target == 2))

    # 1) Full 3-class consistency.
    loss_3 = F.nll_loss(
        log_probs,
        target,
        ignore_index=IGNORE_INDEX,
    )

    # 2) Explicit water-vs-background supervision.
    water_target = ((target == 1) | (target == 2)).float()
    water_pw = torch.tensor(WATER_POS_WEIGHT, device=target.device, dtype=torch.float32)
    loss_water = F.binary_cross_entropy_with_logits(
        water_logit[valid],
        water_target[valid],
        pos_weight=water_pw,
    )

    # 3) Explicit permanent-vs-temporary supervision ONLY on true water pixels.
    type_target = (target == 2).float()
    type_pw = torch.tensor(TYPE_TEMP_POS_WEIGHT, device=target.device, dtype=torch.float32)

    if water_mask.any():
        loss_type = F.binary_cross_entropy_with_logits(
            type_logit[water_mask],
            type_target[water_mask],
            pos_weight=type_pw,
        )
    else:
        loss_type = log_probs.sum() * 0.0

    # 4) Conservative 04E distillation on high-confidence valid pixels.
    # This is the protection mechanism against the 04F permanent-water collapse.
    with torch.no_grad():
        base_conf = base_prob.max(dim=1).values
        anchor_mask = valid & (base_conf >= ANCHOR_CONFIDENCE)

    if anchor_mask.any():
        kl_map = F.kl_div(
            log_probs,
            base_prob,
            reduction="none",
        ).sum(dim=1)
        loss_anchor = kl_map[anchor_mask].mean()
    else:
        loss_anchor = log_probs.sum() * 0.0

    total = (
        LOSS_W_3CLASS * loss_3
        + LOSS_W_WATER * loss_water
        + LOSS_W_TYPE * loss_type
        + LOSS_W_ANCHOR * loss_anchor
    )

    parts = {
        "total": float(total.detach().cpu()),
        "three_class": float(loss_3.detach().cpu()),
        "water": float(loss_water.detach().cpu()),
        "type": float(loss_type.detach().cpu()),
        "anchor": float(loss_anchor.detach().cpu()),
    }
    return total, parts

print("Loss weights:",
      {"3class": LOSS_W_3CLASS, "water": LOSS_W_WATER,
       "type": LOSS_W_TYPE, "anchor": LOSS_W_ANCHOR})
print("P10-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C09 — METRICS + 4-WAY TTA
# ============================================================
def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()

    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(tp, den_iou, out=np.full_like(tp, np.nan), where=den_iou > 0)

    den_dice = 2 * tp + fp + fn
    dice = np.divide(2 * tp, den_dice, out=np.full_like(tp, np.nan), where=den_dice > 0)

    out = {
        "mIoU": float(np.nanmean(iou)),
        "macro_dice": float(np.nanmean(dice)),
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])
        out[f"{name}_Dice"] = float(dice[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(tensor, mode):
    if mode == "identity":
        return tensor
    if mode == "hflip":
        return torch.flip(tensor, dims=[-1])
    if mode == "vflip":
        return torch.flip(tensor, dims=[-2])
    if mode == "rot180":
        return torch.rot90(tensor, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(tensor, mode):
    return _tta_transform(tensor, mode)

@torch.inference_mode()
def tta_logits_batch(model_to_eval, sar, optical, reliability):
    modes = ["identity", "hflip", "vflip", "rot180"]
    total = None

    model_to_eval.eval()

    for mode in modes:
        sar_t = _tta_transform(sar, mode)
        opt_t = _tta_transform(optical, mode)
        rel_t = _tta_transform(reliability, mode)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            logits = model_to_eval(sar_t, opt_t, rel_t, return_aux=False)

        logits = _tta_inverse(logits, mode).float()
        total = logits if total is None else total + logits

    return total / len(modes)

@torch.inference_mode()
def evaluate_4way(model_to_eval, loader, collect_delta_stats=False):
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
    delta_abs_water = []
    delta_abs_type = []

    model_to_eval.eval()

    for bi, b in enumerate(loader, start=1):
        sar = b["sar"].to(DEVICE, non_blocking=True)
        opt = b["optical"].to(DEVICE, non_blocking=True)
        rel = b["reliability"].to(DEVICE, non_blocking=True)
        y = b["target"].to(DEVICE, non_blocking=True)

        logits = tta_logits_batch(model_to_eval, sar, opt, rel)
        pred = logits.argmax(1)
        cm += confusion_from_predictions(pred, y).cpu()

        if collect_delta_stats and isinstance(model_to_eval, FloodShift04H):
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                aux = model_to_eval(sar, opt, rel, return_aux=True)
            valid = y != IGNORE_INDEX
            if valid.any():
                delta_abs_water.append(aux["delta_water"][valid].abs().mean().item())
                delta_abs_type.append(aux["delta_type"][valid].abs().mean().item())

        if bi == 1 or bi % 30 == 0 or bi == len(loader):
            print(f"[{bi:03d}/{len(loader)}] validation")

    metrics = metrics_from_confusion(cm)

    delta_stats = None
    if delta_abs_water:
        delta_stats = {
            "mean_abs_delta_water": float(np.mean(delta_abs_water)),
            "mean_abs_delta_type": float(np.mean(delta_abs_type)),
        }

    return metrics, delta_stats

print("P10-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C10 — MANDATORY WARM-START VALIDATION REPRODUCTION
# ============================================================
print("Evaluating frozen 04E reference with 4-way TTA...")
metrics_04e, _ = evaluate_4way(base_04e, val_loader)

print("\nEvaluating zero-init 04H with 4-way TTA...")
metrics_04h_zero, _ = evaluate_4way(model, val_loader)

print("=" * 112)
print("P10-C10 — WARM-START AUDIT")
print("=" * 112)
print("04E      :", compact_metrics(metrics_04e))
print("04H zero :", compact_metrics(metrics_04h_zero))

assert abs(metrics_04e["mIoU"] - 0.7791) < 0.010, "04E validation reproduction failed."

cm_e = np.asarray(metrics_04e["confusion"])
cm_h = np.asarray(metrics_04h_zero["confusion"])
assert np.array_equal(cm_e, cm_h), "Zero-init 04H predictions differ from 04E."

print("Warm-start confusion difference:", int(np.abs(cm_e - cm_h).sum()))
print("P10-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C11 — FINAL CONTROLLED 04H TRAINING
#
# HEAD ONLY. Exactly 6 epochs. No unfreezing. No hyperparameter sweep.
# ============================================================
EPOCHS = 6
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
WARMUP_FRACTION = 0.10

PROMOTION_MIN_MIOU = 0.790
PROMOTION_MIN_TEMP_IOU = 0.570
PROMOTION_MIN_PERM_IOU = 0.820

REF_04G = {
    "mIoU": 0.7953,
    "non_flood": 0.9749,
    "permanent": 0.8339,
    "temporary": 0.5772,
}
SINGLE_MODEL_MIOU_TOLERANCE = 0.003
SINGLE_MODEL_MIN_MIOU = REF_04G["mIoU"] - SINGLE_MODEL_MIOU_TOLERANCE

trainable_params = [p for p in model.parameters() if p.requires_grad]
assert trainable_params and all(not p.requires_grad for p in model.base.parameters())

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = max(1, steps_per_epoch * EPOCHS)
warmup_steps = max(1, int(total_steps * WARMUP_FRACTION))

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

history = []
best_miou = -float("inf")
best_gate_miou = -float("inf")
best_miou_path = CHECKPOINT_ROOT / "FloodShift_04H_best_mIoU.pt"
best_gate_path = CHECKPOINT_ROOT / "FloodShift_04H_best_gate.pt"
head_only_path = CHECKPOINT_ROOT / "FloodShift_04H_head_only.pt"

nonfinite_batches = 0
optimizer_steps = 0
train_start = time.time()

def save_checkpoint(path, epoch, val_metrics, tag):
    payload = {
        "experiment": "FloodShift-04H-Hierarchical-Single-Model",
        "tag": tag,
        "epoch": int(epoch),
        "base_checkpoint": str(CHECKPOINT_04E),
        "base_sha256": sha04e,
        "optional_04f_sha256": sha04f,
        "state_dict": model.state_dict(),
        "head_state_dict": model.head.state_dict(),
        "validation_metrics": val_metrics,
        "config": {
            "epochs": EPOCHS,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
            "head_lr": HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
            "max_grad_norm": MAX_GRAD_NORM,
            "warmup_fraction": WARMUP_FRACTION,
            "water_pos_weight": WATER_POS_WEIGHT,
            "type_temp_pos_weight": TYPE_TEMP_POS_WEIGHT,
            "loss_weights": {
                "three_class": LOSS_W_3CLASS,
                "water": LOSS_W_WATER,
                "type": LOSS_W_TYPE,
                "anchor": LOSS_W_ANCHOR,
            },
            "anchor_confidence": ANCHOR_CONFIDENCE,
            "seed": SEED,
            "tta": ["identity", "hflip", "vflip", "rot180"],
        },
    }
    torch.save(payload, path)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    model.train()
    model.base.eval()
    optimizer.zero_grad(set_to_none=True)

    running = {
        "total": 0.0,
        "three_class": 0.0,
        "water": 0.0,
        "type": 0.0,
        "anchor": 0.0,
    }
    counted_batches = 0

    for bi, b in enumerate(train_loader, start=1):
        sar = b["sar"].to(DEVICE, non_blocking=True)
        opt = b["optical"].to(DEVICE, non_blocking=True)
        rel = b["reliability"].to(DEVICE, non_blocking=True)
        y = b["target"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            outputs = model(sar, opt, rel, return_aux=True)
            loss, parts = hierarchical_loss(outputs, y)

        if not torch.isfinite(loss):
            nonfinite_batches += 1
            print(f"⚠️ non-finite loss at epoch {epoch}, batch {bi}; batch skipped")
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()

        for k in running:
            running[k] += parts[k]
        counted_batches += 1

        should_step = (bi % GRAD_ACCUM_STEPS == 0) or (bi == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            optimizer_steps += 1

        if bi == 1 or bi % 100 == 0 or bi == len(train_loader):
            print(
                f"Epoch {epoch}/{EPOCHS} "
                f"[{bi:03d}/{len(train_loader)}] "
                f"loss={parts['total']:.4f} "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

    # Exact same 4-way TTA validation protocol as 04E/04F/04G.
    val_metrics, delta_stats = evaluate_4way(model, val_loader, collect_delta_stats=True)

    train_means = {
        k: (v / max(counted_batches, 1))
        for k, v in running.items()
    }

    row = {
        "epoch": epoch,
        "train": train_means,
        "validation": val_metrics,
        "compact_validation": compact_metrics(val_metrics),
        "delta_stats": delta_stats,
        "lr_end": optimizer.param_groups[0]["lr"],
        "epoch_minutes": (time.time() - epoch_start) / 60.0,
    }
    history.append(row)

    print("=" * 112)
    print(f"EPOCH {epoch} SUMMARY")
    print("Train :", {k: round(v, 4) for k, v in train_means.items()})
    print("Val   :", compact_metrics(val_metrics))
    print("Delta :", delta_stats)
    print("Time  :", round(row["epoch_minutes"], 2), "min")
    print("=" * 112)

    # Track global best mIoU.
    if val_metrics["mIoU"] > best_miou:
        best_miou = val_metrics["mIoU"]
        save_checkpoint(best_miou_path, epoch, val_metrics, "best_mIoU")
        print("Saved global best:", best_miou_path)

    # Track best model that already satisfies the predeclared scientific gate.
    gate_ok = (
        val_metrics["mIoU"] >= PROMOTION_MIN_MIOU
        and val_metrics["temporary_flood_IoU"] >= PROMOTION_MIN_TEMP_IOU
        and val_metrics["permanent_water_IoU"] >= PROMOTION_MIN_PERM_IOU
    )

    if gate_ok and val_metrics["mIoU"] > best_gate_miou:
        best_gate_miou = val_metrics["mIoU"]
        save_checkpoint(best_gate_path, epoch, val_metrics, "best_gate")
        print("Saved gate-feasible best:", best_gate_path)

    with open(HISTORY_ROOT / "FloodShift_04H_history.json", "w") as f:
        json.dump(history, f, indent=2)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTraining minutes:", round((time.time() - train_start) / 60, 2))
print("Optimizer steps :", optimizer_steps)
print("Nonfinite batches:", nonfinite_batches)

# Save the head separately for compact archival/reconstruction.
torch.save(
    {
        "experiment": "FloodShift-04H-Hierarchical-Single-Model",
        "base_sha256": sha04e,
        "head_state_dict": model.head.state_dict(),
        "history": history,
    },
    head_only_path,
)

assert nonfinite_batches == 0, "04H encountered non-finite training batches."
print("P10-C11 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P10-C12 — FINAL VALIDATION SELECTION + PROMOTION DECISION
# NO TEST SET IS ACCESSED HERE.
# ============================================================
selected_path = best_gate_path if best_gate_path.exists() else best_miou_path
assert selected_path.exists(), "No 04H checkpoint was saved."

payload = torch.load(selected_path, map_location="cpu", weights_only=False)
state = payload["state_dict"]
missing, unexpected = model.load_state_dict(state, strict=False)
assert not missing, f"Selected 04H missing keys: {missing[:10]}"
assert not unexpected, f"Selected 04H unexpected keys: {unexpected[:10]}"
model = model.to(DEVICE).eval()

print("Selected checkpoint:", selected_path)
print("Selected epoch     :", payload.get("epoch"))
print("Selection tag      :", payload.get("tag"))

final_metrics, final_delta_stats = evaluate_4way(
    model,
    val_loader,
    collect_delta_stats=True,
)

compact_final = compact_metrics(final_metrics)

gate_pass = (
    final_metrics["mIoU"] >= PROMOTION_MIN_MIOU
    and final_metrics["temporary_flood_IoU"] >= PROMOTION_MIN_TEMP_IOU
    and final_metrics["permanent_water_IoU"] >= PROMOTION_MIN_PERM_IOU
)

single_model_preferred = (
    gate_pass
    and final_metrics["mIoU"] >= SINGLE_MODEL_MIN_MIOU
)

print("=" * 112)
print("P10-C12 — FINAL 04H VALIDATION")
print("=" * 112)
print("04E reference :", compact_metrics(metrics_04e))
print("04G reference :", REF_04G)
print("04H final     :", compact_final)
print("Residual stats:", final_delta_stats)
print()
print(
    f"Scientific gate: mIoU>={PROMOTION_MIN_MIOU:.3f}, "
    f"temporary>={PROMOTION_MIN_TEMP_IOU:.3f}, "
    f"permanent>={PROMOTION_MIN_PERM_IOU:.3f}"
)
print(
    f"Single-model competitiveness: mIoU >= {SINGLE_MODEL_MIN_MIOU:.4f} "
    f"(within {SINGLE_MODEL_MIOU_TOLERANCE:.3f} of 04G)"
)

if single_model_preferred:
    decision = "04H_SINGLE_MODEL_PREFERRED"
    print("\nDECISION: 04H SINGLE MODEL PREFERRED ✅")
    print("Keep frozen test LOCKED until robustness/uncertainty protocol is frozen.")
elif gate_pass:
    decision = "04H_VALID_BUT_04G_VAL_CHAMPION"
    print("\nDECISION: 04H PASSES THE SCIENTIFIC GATE ✅")
    print("But it is not within the predeclared tolerance of 04G's validation mIoU.")
    print("Do not open the frozen test yet; compare deployment/scientific trade-off first.")
else:
    decision = "04H_NOT_PROMOTED"
    print("\nDECISION: 04H NOT PROMOTED ❌")
    print("Do NOT open the frozen test. Do NOT automatically launch 04I.")

print("\n🔒 FROZEN TEST REMAINS LOCKED IN ALL CASES.")


In [ ]:
# ============================================================
# CELL: P10-C13 — EXPORT FINAL 04H RECORD + COMPACT ARCHIVE
# ============================================================
record = {
    "experiment": "FloodShift-04H-Hierarchical-Single-Model",
    "purpose": (
        "single-model hierarchical water/non-water and permanent/temporary refinement "
        "derived from the 04G validation finding"
    ),
    "base_04e": {
        "checkpoint": str(CHECKPOINT_04E),
        "sha256": sha04e,
        "validation_4way_tta": metrics_04e,
    },
    "optional_04f_audit": {
        "checkpoint": str(CHECKPOINT_04F) if CHECKPOINT_04F else None,
        "sha256": sha04f,
        "used_for_training": False,
    },
    "reference_04g": REF_04G,
    "training": {
        "epochs": EPOCHS,
        "head_only": True,
        "terraMind_rift_decoder_frozen": True,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "nonfinite_batches": nonfinite_batches,
        "train_class_counts": {
            "non_flood": int(train_counts[0]),
            "permanent_water": int(train_counts[1]),
            "temporary_flood": int(train_counts[2]),
        },
        "water_pos_weight": WATER_POS_WEIGHT,
        "type_temp_pos_weight": TYPE_TEMP_POS_WEIGHT,
        "history": history,
    },
    "selection": {
        "selected_checkpoint": str(selected_path),
        "selected_epoch": int(payload.get("epoch", -1)),
        "selected_tag": payload.get("tag"),
    },
    "validation": {
        "04h_final": final_metrics,
        "04h_compact": compact_final,
        "delta_stats": final_delta_stats,
        "gate": {
            "mIoU": PROMOTION_MIN_MIOU,
            "temporary": PROMOTION_MIN_TEMP_IOU,
            "permanent": PROMOTION_MIN_PERM_IOU,
        },
        "single_model_min_mIoU": SINGLE_MODEL_MIN_MIOU,
        "gate_pass": bool(gate_pass),
        "single_model_preferred": bool(single_model_preferred),
        "decision": decision,
    },
    "test_status": "LOCKED_NOT_EVALUATED",
}

record_path = OUTPUT_ROOT / "FloodShift_04H_decision_record.json"
with open(record_path, "w") as f:
    json.dump(record, f, indent=2)

# Compact archive: metadata + head-only checkpoint + history.
# The full selected checkpoint remains separately available in /kaggle/working.
archive_path = OUTPUT_ROOT / "FloodShift_04H_COMPACT_ARCHIVE.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [
        record_path,
        HISTORY_ROOT / "FloodShift_04H_history.json",
        head_only_path,
    ]:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)

print("=" * 112)
print("P10-C13 — EXPORT")
print("=" * 112)
print("Decision record :", record_path)
print("Compact archive :", archive_path)
print("Best mIoU ckpt  :", best_miou_path if best_miou_path.exists() else "none")
print("Best gate ckpt  :", best_gate_path if best_gate_path.exists() else "none")
print("Selected ckpt   :", selected_path)
print("Decision        :", decision)
print("Test            : LOCKED_NOT_EVALUATED")
print("P10-C13 COMPLETE ✅")


## After the run

Send back the full outputs from **P10-C11, P10-C12, and P10-C13**.

Do **not** run the 04G frozen-test cell after this notebook. We first decide whether 04H is a credible single-model replacement and then freeze the robustness/uncertainty protocol before the one final geographic test.

### Fixed 04H configuration

- GPU: one T4
- 04E TerraMind-RIFT: **fully frozen**
- Trainable component: **hierarchical residual head only**
- Epochs: **6**
- Train batch: **2**
- Gradient accumulation: **4**
- Effective batch: **8**
- Optimizer: **AdamW**
- Head LR: **3e-4**
- Weight decay: **1e-4**
- Warm-up: **10%**, then cosine decay
- AMP: enabled on CUDA
- Gradient clipping: **1.0**
- Augmentation: frozen D4-aligned flips/rotations
- Validation: **4-way TTA**
- Frozen test: **never constructed/evaluated**

The notebook intentionally contains **no automatic fallback training stage and no backbone unfreezing**.
